In [1]:
import os
import pandas as pd

# 1. Load Bisola's raw daily rainfall file (auto-handles notebooks/ or root folder)
raw_path = "../data/raw/Northern_Rainfall.csv" if os.path.exists("../data/raw/Northern_Rainfall.csv") else "data/raw/Northern_Rainfall.csv"
df = pd.read_csv(raw_path)

# 2. Standardize dates to YYYY-MM-01
df["date"] = pd.to_datetime(df["Date"]).dt.to_period("M").dt.to_timestamp()

# 3. Sum daily precipitation into monthly totals per city, then average across the 3 hubs
city_cols = ["Kano_Precipitation (mm)", "Kaduna_Precipitation (mm)", "Jos_Precipitation (mm)"]
monthly_city_totals = df.groupby("date")[city_cols].sum()
monthly_city_totals["north_rainfall_mm"] = monthly_city_totals.mean(axis=1).round(2)

# 4. Align strictly to all 72 months (2020-01-01 to 2025-12-01)
full_months = pd.DataFrame({"date": pd.date_range("2020-01-01", "2025-12-01", freq="MS")})
rain_clean = full_months.merge(monthly_city_totals[["north_rainfall_mm"]].reset_index(), on="date", how="left")

# 5. Save cleaned monthly rainfall dataset
out_dir = "../data/processed" if os.path.exists("../data") else "data/processed"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "clean_rainfall.csv")
rain_clean.to_csv(out_path, index=False)

print("Done! Saved:", out_path)
print("Total rows:", len(rain_clean), "| Missing values:", rain_clean["north_rainfall_mm"].isnull().sum())
rain_clean.head(6)

Done! Saved: ../data/processed\clean_rainfall.csv
Total rows: 72 | Missing values: 0


,date,north_rainfall_mm
0,2020-01-01,0.00
1,2020-02-01,0.00
2,2020-03-01,2.17
3,2020-04-01,5.00
4,2020-05-01,33.87
5,2020-06-01,68.77
